## Running hyperparameter optimization - Part 2

This notebook continues the hyperparameter tuning of MOVE. In Part 1 (notebook 02)
we looked at how well models reconstruct the training and test sets. Here we
focus on the **stability** of the latent space: if we train the same model
several times, do we get a similar latent space each time?

For each hyperparameter combination, MOVE trains `num_refits` models. The first
model is used as a baseline: we compute the cosine similarity between every pair
of samples in its latent space. For each of the other models, we compute the
same similarities and take the mean absolute difference from the baseline. This
difference is the stability score, so **lower values mean a more stable latent
space**.

⚠️ **Before you start:** this notebook uses the encoded data in `interim_data/`,
which is created by notebook 01 (Encoding data). If you are starting from a fresh
clone of the repository, run notebook 01 first (it only takes a few seconds).

In [2]:
# Run from the tutorial folder, so relative paths (config/, data/,
# interim_data/, results/) resolve no matter where the notebook was started.
import os
from pathlib import Path

if not Path("config").is_dir() and Path("../config").is_dir():
    os.chdir("..")

As in Part 1, it's best to run this task from the command line, because many
models are trained*:

\* Note: the command may take a while. The experiment trains 12 hyperparameter
combinations × 10 refits = 120 models. To make it faster, lower the number of
refits, e.g. `move-dl experiment=random_small__tune_stability task.num_refits=3`.

In [ ]:
! move-dl experiment=random_small__tune_stability

The output of the previous command is a CSV table with one row per
hyperparameter combination, summarizing the stability scores of its refits
(mean, median, quartiles, and whiskers).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

results = pd.read_csv("results/tuning/stability.csv")

# Columns with the swept hyperparameters (e.g., "task.batch_size")
param_cols = [col for col in results.columns if col.startswith("task.")]

results

We can plot the distribution of stability scores of each hyperparameter
combination, sorted from most to least stable.

In [ ]:
def combo_label(row: pd.Series) -> str:
    return ", ".join(f"{col.split('.')[-1]}={row[col]}" for col in param_cols)

plot_data = results.sort_values("med")
stats = [
    {
        "label": combo_label(row),
        **{key: row[key] for key in ["med", "q1", "q3", "whislo", "whishi", "mean"]},
    }
    for _, row in plot_data.iterrows()
]

fig, ax = plt.subplots(figsize=(6, 0.4 * len(stats) + 1))
ax.bxp(stats, vert=False, showmeans=True, showfliers=False, patch_artist=True, boxprops=dict(facecolor="#7570b3"))
ax.invert_yaxis()  # most stable on top
ax.set(xlabel="Stability (mean abs. difference in cosine similarity, lower is better)")
plt.show()

Stability alone is not enough to pick a model: a model that barely learns
anything can be very stable. So we combine it with the reconstruction results
from Part 1 (if you ran notebook 02 with the same hyperparameter grid), using
the median test-set reconstruction score averaged across datasets.

In [ ]:
summary = results[param_cols + ["med"]].rename(columns={"med": "stability"})

metrics_path = Path("results/tuning/metrics.csv")
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    test_reconstruction = (
        metrics[metrics["split"] == "test"]
        .groupby(param_cols, as_index=False)["med"]
        .mean()
        .rename(columns={"med": "test_reconstruction"})
    )
    summary = summary.merge(test_reconstruction, on=param_cols, how="left")
else:
    print(f"'{metrics_path}' not found: run notebook 02 to add reconstruction scores.")

summary.sort_values("stability").reset_index(drop=True)

Choose a hyperparameter combination that has both a low stability score and a
high test reconstruction score. Then set those values in the task configs used
in the next notebooks, under `model_config` and `training_loop_config`:

- `config/task/random_small__latent.yaml` (notebook 04)
- `config/task/random_small__id_assoc_bayes.yaml` (notebook 05)